# RCPGenerator — Getting Started

RCPGenerator generates random close packings of polydisperse spheres in N dimensions — 2D disks, 3D spheres, and hyperspheres — in **flat (periodic / hard-wall) and curved (disk / cylinder / sphere) containers**. The core is a parallel C++/OpenMP + SIMD engine with a stateful Python API; this release is many times faster than the original with improved convergence, and the API and results are unchanged.

This notebook installs the package, walks a gallery of packings (box, curved-container, and target-φ staging), sets the thread count, and shows how to **save a packing and open it in the interactive web viewer**. It ships two importable packages: `rcpgenerator` (the engine) and `rcptools` (search / analysis / bundle helpers).


## Installation

Clone the repository, move into the Python package, and build the C++ core + bindings (a minute or two).


In [ ]:
!git clone https://github.com/KD-physics/RCPGenerator.git
%cd RCPGenerator/python_code/python
!pip install -v .


## Verify the install

Confirm the engine and the companion toolbox both import.


In [ ]:
import rcpgenerator, rcptools
print(rcpgenerator.Packing)
print('rcptools ready:', hasattr(rcptools, 'run_one_custom_packing'))


## Thread count

The engine is parallel (OpenMP). Set the worker threads once — leave it at the default to use all cores, or cap it on a shared machine.


In [ ]:
rcpgenerator.set_num_threads(2)          # free Colab has 2 CPUs; raise this on a bigger machine
print('threads:', rcpgenerator.get_num_threads())


## Timing the packer

The `pack()` call is fast; **rendering** is the slow part. So every example below times only the pack and prints it, then draws the packing in a **separate** cell — that way the printed time reflects the packer, not matplotlib.


## 2D disks — monodisperse, periodic box

The simplest case: equal disks in a fully periodic square.


In [ ]:
import time
t0 = time.perf_counter()
p = rcpgenerator.Packing(phi=0.11, N=250, Ndim=2, box=[1., 1.], walls=[0, 0],
                         dist={'type': 'mono', 'd': 1.0},
                         fix_height=False, neighbor_max=0, seed=123)
p.pack()
print(f'packed N={p.N} in {time.perf_counter()-t0:.2f} s   ->   phi = {p.phi_final:.3f},  steps = {p.steps}')


In [ ]:
p.show_packing(palette_choice=1)   # rendering is the slow part — kept in its own cell


## 2D disks — bidisperse, periodic box

Two disk sizes mixed 50/50 — the classic glass-former mixture.


In [ ]:
import time
t0 = time.perf_counter()
p = rcpgenerator.Packing(phi=0.11, N=250, Ndim=2, box=[1., 1.], walls=[0, 0],
                         dist={'type': 'bidisperse', 'd1': 0.8, 'd2': 1.2, 'p': 0.5},
                         fix_height=False, neighbor_max=0, seed=124)
p.pack()
print(f'packed N={p.N} in {time.perf_counter()-t0:.2f} s   ->   phi = {p.phi_final:.3f},  steps = {p.steps}')


In [ ]:
p.show_packing(palette_choice=5)   # rendering is the slow part — kept in its own cell


## 2D disks — lognormal polydispersity, periodic box

A continuous size distribution (lognormal), the common model for real powders.


In [ ]:
import time
t0 = time.perf_counter()
p = rcpgenerator.Packing(phi=0.11, N=250, Ndim=2, box=[1., 1.], walls=[0, 0],
                         dist={'type': 'lognormal', 'mu': 0.0, 'sigma': 0.35},
                         fix_height=False, neighbor_max=0, seed=125)
p.pack()
print(f'packed N={p.N} in {time.perf_counter()-t0:.2f} s   ->   phi = {p.phi_final:.3f},  steps = {p.steps}')


In [ ]:
p.show_packing(palette_choice=10)   # rendering is the slow part — kept in its own cell


## Your own particle sizes (custom distribution)

Beyond the built-in families, pass **any** array of `N` diameters via `dist={"type": "custom", "custom": [...]}` — handy for a measured or hand-built size distribution. Here a trimodal mix of 200 large, 150 medium, and 50 small disks.


In [ ]:
import time, numpy as np
sizes = np.concatenate([np.full(200, 1.0), np.full(150, 0.6), np.full(50, 0.3)])   # your own N diameters
t0 = time.perf_counter()
p = rcpgenerator.Packing(phi=0.11, N=len(sizes), Ndim=2, box=[1., 1.], walls=[0, 0],
                         fix_height=False, dist={'type': 'custom', 'custom': sizes.tolist()},
                         neighbor_max=0, seed=42)
p.pack()
print(f'packed N={p.N} custom sizes in {time.perf_counter()-t0:.2f} s   ->   phi = {p.phi_final:.3f},  steps = {p.steps}')


In [ ]:
p.show_packing(palette_choice=7)   # rendering is the slow part — kept in its own cell


## 2D disks in a circular container (hard wall)

A hard circular boundary: walls=[-2, 0] makes the first two dims share one disk-shaped wall of diameter box[0].


In [ ]:
import time
t0 = time.perf_counter()
p = rcpgenerator.Packing(phi=0.25, N=500, Ndim=2, box=[1., 1.], walls=[-2, 0],
                         dist={'type': 'mono', 'd': 1.0},
                         fix_height=False, neighbor_max=0, seed=128)
p.pack()
print(f'packed N={p.N} in {time.perf_counter()-t0:.2f} s   ->   phi = {p.phi_final:.3f},  steps = {p.steps}')


In [ ]:
p.show_packing(palette_choice=12)   # rendering is the slow part — kept in its own cell


## 3D spheres — monodisperse, periodic box

Equal spheres in a periodic cube — random close packing lands near φ ≈ 0.64.


In [ ]:
import time
t0 = time.perf_counter()
p = rcpgenerator.Packing(phi=0.08, N=250, Ndim=3, box=[1., 1., 1.], walls=[0, 0, 0],
                         dist={'type': 'mono', 'd': 1.0},
                         fix_height=False, neighbor_max=0, seed=126)
p.pack()
print(f'packed N={p.N} in {time.perf_counter()-t0:.2f} s   ->   phi = {p.phi_final:.3f},  steps = {p.steps}')


In [ ]:
p.show_packing(palette_choice=2)   # rendering is the slow part — kept in its own cell


## 3D spheres — bidisperse, periodic box

A two-size sphere mixture in a periodic cube.


In [ ]:
import time
t0 = time.perf_counter()
p = rcpgenerator.Packing(phi=0.08, N=250, Ndim=3, box=[1., 1., 1.], walls=[0, 0, 0],
                         dist={'type': 'bidisperse', 'd1': 0.85, 'd2': 1.15, 'p': 0.5},
                         fix_height=False, neighbor_max=0, seed=127)
p.pack()
print(f'packed N={p.N} in {time.perf_counter()-t0:.2f} s   ->   phi = {p.phi_final:.3f},  steps = {p.steps}')


In [ ]:
p.show_packing(palette_choice=6)   # rendering is the slow part — kept in its own cell


## 3D spheres in a cylinder (hard wall)

A cylindrical hard wall: walls=[-2, 0, 0] curves the first two dims (the cross-section) and leaves the axis periodic.


In [ ]:
import time
t0 = time.perf_counter()
p = rcpgenerator.Packing(phi=0.25, N=500, Ndim=3, box=[1., 1., 1.], walls=[-2, 0, 0],
                         dist={'type': 'mono', 'd': 1.0},
                         fix_height=False, neighbor_max=0, seed=129)
p.pack()
print(f'packed N={p.N} in {time.perf_counter()-t0:.2f} s   ->   phi = {p.phi_final:.3f},  steps = {p.steps}')


In [ ]:
p.show_packing(palette_choice=9)   # rendering is the slow part — kept in its own cell


## 3D spheres in a sphere (hard wall)

A spherical hard container: walls=[-3, 0, 0] curves all three dims into one sphere of diameter box[0].


In [ ]:
import time
t0 = time.perf_counter()
p = rcpgenerator.Packing(phi=0.25, N=500, Ndim=3, box=[1., 1., 1.], walls=[-3, 0, 0],
                         dist={'type': 'mono', 'd': 1.0},
                         fix_height=False, neighbor_max=0, seed=130)
p.pack()
print(f'packed N={p.N} in {time.perf_counter()-t0:.2f} s   ->   phi = {p.phi_final:.3f},  steps = {p.steps}')


In [ ]:
p.show_packing(palette_choice=4)   # rendering is the slow part — kept in its own cell


## Pushing to a target packing fraction (φ-staging)

Grow to a target φ, then step it higher while relaxing at fixed diameter — a simple way to drive a power-law packing toward a chosen density.


In [ ]:
import time
t0 = time.perf_counter()
p = rcpgenerator.Packing(phi=0.11, N=250, Ndim=2, box=[1., 1.], walls=[0, 0],
                         dist={'type': 'powerlaw', 'd_min': 0.3, 'd_max': 1.8, 'exponent': -2.5},
                         fix_height=False, neighbor_max=0, seed=131)
p.relax(n_steps=5000, target_phi=0.82)                 # grow to a target phi
for target in [0.84, 0.86, 0.88]:                      # then step higher, relaxing at fixed diameter
    p.update_phi(target); p.relax(n_steps=1000, fix_diameter=True)
print(f'staged to phi = {p.phi_final:.3f} in {time.perf_counter()-t0:.2f} s')


In [ ]:
p.show_packing(palette_choice=8)   # rendering is the slow part — kept in its own cell


## Save a packing and view it interactively

`rcptools` writes a **bundle** — the format the web viewer reads (`manifest.json` + raw `pos.f32` / `dia.f32`). Here we pack a sphere-in-a-sphere and save it, then open it in the viewer to rotate, slice, and colour it.


In [ ]:
from rcptools.bridge import write_bundle
import numpy as np, os

sp = rcpgenerator.Packing(phi=0.25, N=500, Ndim=3, box=[1., 1., 1.], walls=[-3, 0, 0],
                          fix_height=False, dist={'type': 'mono', 'd': 1.0}, neighbor_max=0, seed=7)
sp.pack()
write_bundle('bundles/my_sphere', np.asarray(sp.positions), np.asarray(sp.diameters),
             list(sp.box), walls=list(sp.walls))
print('wrote bundles/my_sphere/ ->', sorted(os.listdir('bundles/my_sphere')))


**View it.** The interactive viewer ships with the package (`webapp/`). Build a self-contained HTML and open it in any browser:

```bash
python webapp/launch.py bundles/my_sphere            # builds + opens it locally
# in Colab (no local browser): add --no-open --out my_sphere.html, then download my_sphere.html
```

A **solid** boundary is a hard wall and a **dashed** boundary is periodic; for a sphere the cross-section circle grows and shrinks as you scrub through z. (You can also serve `webapp/` with a local web server and load the `bundles/my_sphere` folder via the *Choose bundle folder* button.)
